In [1]:
import pandas as pd
import sys
import os
from pathlib import Path
import numpy as np

# Lấy đường dẫn thư mục gốc (nơi chứa src/)
project_root = Path.cwd().parent  # Đi lên 1 cấp từ notebook_file/
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))


df = pd.read_parquet("../data/universe/ohlcv_daily.parquet")

df

,timestamp,open,high,low,close,volume,symbol,dollar_volume,median_dollar_volume_30d,first_trading_date,listing_age_days,listing_eligible
0,2020-01-01 00:00:00+00:00,7195.24,7255.00,7175.15,7200.85,16792.388165,BTC/USDT,1.209195e+08,NaN,2020-01-01 00:00:00+00:00,0,False
1,2020-01-02 00:00:00+00:00,7200.77,7212.50,6924.74,6965.71,31951.483932,BTC/USDT,2.225648e+08,NaN,2020-01-01 00:00:00+00:00,1,False
2,2020-01-03 00:00:00+00:00,6965.49,7405.00,6871.04,7344.96,68428.500451,BTC/USDT,5.026046e+08,NaN,2020-01-01 00:00:00+00:00,2,False
3,2020-01-04 00:00:00+00:00,7345.00,7404.00,7272.21,7354.11,29987.974977,BTC/USDT,2.205349e+08,NaN,2020-01-01 00:00:00+00:00,3,False
4,2020-01-05 00:00:00+00:00,7354.19,7495.00,7318.00,7358.75,38331.085604,BTC/USDT,2.820689e+08,NaN,2020-01-01 00:00:00+00:00,4,False
...,...,...,...,...,...,...,...,...,...,...,...,...
524146,2026-08-13 00:00:00+00:00,18.52,18.81,18.39,18.63,28117.670000,GMEB/USDT,5.238322e+05,NaN,2026-08-12 00:00:00+00:00,1,False
524147,2026-08-14 00:00:00+00:00,18.65,18.82,18.49,18.70,25419.860000,GMEB/USDT,4.753514e+05,NaN,2026-08-12 00:00:00+00:00,2,False
524148,2026-08-15 00:00:00+00:00,18.69,18.86,18.69,18.79,11029.660000,GMEB/USDT,2.072473e+05,NaN,2026-08-12 00:00:00+00:00,3,False
524149,2026-08-16 00:00:00+00:00,18.78,18.85,18.66,18.76,9054.200000,GMEB/USDT,1.698568e+05,NaN,2026-08-12 00:00:00+00:00,4,False


# A: Data Infrastructure & Universe Construction 

Ngày t, những coin nào được phép thuộc investment universe

In [2]:
df[df["symbol"] == "TWT/USDT"].head(10)

,timestamp,open,high,low,close,volume,symbol,dollar_volume,median_dollar_volume_30d,first_trading_date,listing_age_days,listing_eligible
248921,2021-01-27 00:00:00+00:00,0.1698,0.5000,0.1698,0.4310,3.076846e+08,TWT/USDT,1.326121e+08,NaN,2021-01-27 00:00:00+00:00,0,False
248922,2021-01-28 00:00:00+00:00,0.4312,0.4314,0.2919,0.3012,1.566589e+08,TWT/USDT,4.718565e+07,NaN,2021-01-27 00:00:00+00:00,1,False
248923,2021-01-29 00:00:00+00:00,0.3009,0.3210,0.2520,0.2880,7.461134e+07,TWT/USDT,2.148806e+07,NaN,2021-01-27 00:00:00+00:00,2,False
248924,2021-01-30 00:00:00+00:00,0.2878,0.3480,0.2830,0.2980,4.797714e+07,TWT/USDT,1.429719e+07,NaN,2021-01-27 00:00:00+00:00,3,False
248925,2021-01-31 00:00:00+00:00,0.2980,0.3213,0.2833,0.2907,2.659305e+07,TWT/USDT,7.730600e+06,NaN,2021-01-27 00:00:00+00:00,4,False
248926,2021-02-01 00:00:00+00:00,0.2907,0.3197,0.2867,0.3018,2.147512e+07,TWT/USDT,6.481191e+06,NaN,2021-01-27 00:00:00+00:00,5,False
248927,2021-02-02 00:00:00+00:00,0.3018,0.3399,0.2963,0.3266,2.531728e+07,TWT/USDT,8.268622e+06,NaN,2021-01-27 00:00:00+00:00,6,False
248928,2021-02-03 00:00:00+00:00,0.3266,0.3780,0.3265,0.3696,3.085874e+07,TWT/USDT,1.140539e+07,NaN,2021-01-27 00:00:00+00:00,7,False
248929,2021-02-04 00:00:00+00:00,0.3696,0.4405,0.3400,0.4130,4.280897e+07,TWT/USDT,1.768011e+07,NaN,2021-01-27 00:00:00+00:00,8,False
248930,2021-02-05 00:00:00+00:00,0.4127,0.5500,0.4127,0.4634,6.389926e+07,TWT/USDT,2.961092e+07,NaN,2021-01-27 00:00:00+00:00,9,False


In [3]:
df['median_dollar_volume_30d'].describe()

# Đếm số coin > $10M
liquid_coins = df[df['median_dollar_volume_30d'] >= 10000000]
print(f"Number of liquid coins: {liquid_coins['symbol'].nunique()}")

Number of liquid coins: 359


In [4]:
from src.config import STABLECOINS

def build_universe(df, stable_coins = STABLECOINS):
    
    result = df.copy()
    result = result[result["listing_eligible"] == True]
    
    result["is_stable_coins"] = result["symbol"].replace("/USDT","").isin(stable_coins)
    result = result[result["is_stable_coins"] == False]
    
        
    result["is_qualified"] = np.where(result["median_dollar_volume_30d"] >= 10000000,1,0) #có đủ điều kiện thanh khoản cơ bản k
    
    
    return result.reset_index(drop = True)

def is_bad_token(symbol):
    base = symbol.replace('/USDT', '')
    
    patterns = ['UP', 'DOWN', 'BULL', 'BEAR', '3L', '3S', 'HEDGE']
    for p in patterns:
        if p in base:
            return True
    
    blacklist = ['WBTC', 'WETH', 'WSTETH', 'WBETH', 'BTCST', 'ETH2']
    if base in blacklist:
        return True
    
    # 3. Number suffix (VD: BTC1000, ETH1000)
    import re
    if re.search(r'\d+$', base):
        return True
    
    return False  

result = build_universe(df)
result["is_bad"]= result["symbol"].apply(is_bad_token)
result = result[~result["is_bad"]]

result

,timestamp,open,high,low,close,volume,symbol,dollar_volume,median_dollar_volume_30d,first_trading_date,listing_age_days,listing_eligible,is_stable_coins,is_qualified,is_bad
0,2020-06-29 00:00:00+00:00,9116.16000,9238.00000,9024.67000,9192.56000,4.212029e+04,BTC/USDT,3.871933e+08,4.582236e+08,2020-01-01 00:00:00+00:00,180,True,False,1,False
1,2020-06-30 00:00:00+00:00,9192.93000,9205.00000,9064.89000,9138.55000,3.146316e+04,BTC/USDT,2.875277e+08,4.557183e+08,2020-01-01 00:00:00+00:00,181,True,False,1,False
2,2020-07-01 00:00:00+00:00,9138.08000,9292.00000,9080.10000,9232.00000,3.848853e+04,BTC/USDT,3.553261e+08,4.522662e+08,2020-01-01 00:00:00+00:00,182,True,False,1,False
3,2020-07-02 00:00:00+00:00,9231.99000,9261.96000,8940.00000,9086.54000,4.572517e+04,BTC/USDT,4.154836e+08,4.484277e+08,2020-01-01 00:00:00+00:00,183,True,False,1,False
4,2020-07-03 00:00:00+00:00,9086.54000,9125.00000,9037.47000,9058.26000,2.894342e+04,BTC/USDT,2.621770e+08,4.345655e+08,2020-01-01 00:00:00+00:00,184,True,False,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
448703,2026-08-13 00:00:00+00:00,0.07151,0.07935,0.07144,0.07527,4.111634e+07,ESP/USDT,3.094827e+06,1.936905e+06,2026-02-12 00:00:00+00:00,182,True,False,0,False
448704,2026-08-14 00:00:00+00:00,0.07531,0.07696,0.07203,0.07346,1.632267e+07,ESP/USDT,1.199063e+06,2.021874e+06,2026-02-12 00:00:00+00:00,183,True,False,0,False
448705,2026-08-15 00:00:00+00:00,0.07346,0.07512,0.07156,0.07248,1.140049e+07,ESP/USDT,8.263074e+05,2.021874e+06,2026-02-12 00:00:00+00:00,184,True,False,0,False
448706,2026-08-16 00:00:00+00:00,0.07248,0.07313,0.06910,0.07116,9.949239e+06,ESP/USDT,7.079878e+05,2.021874e+06,2026-02-12 00:00:00+00:00,185,True,False,0,False


In [5]:
result["symbol"].unique()

<ArrowStringArray>
[  'BTC/USDT',   'ETH/USDT',   'BNB/USDT',   'NEO/USDT',   'LTC/USDT',
  'QTUM/USDT',   'ADA/USDT',   'XRP/USDT',  'IOTA/USDT',   'XLM/USDT',
 ...
  'BREV/USDT',  '币安人生/USDT',   'ZKP/USDT',     'U/USDT',  'FRAX/USDT',
  'FOGO/USDT', 'RLUSD/USDT',  'SENT/USDT',  'ZAMA/USDT',   'ESP/USDT']
Length: 386, dtype: str

In [6]:
result.drop(columns=["is_stable_coins","is_bad"], inplace = True)
result.head()

,timestamp,open,high,low,close,volume,symbol,dollar_volume,median_dollar_volume_30d,first_trading_date,listing_age_days,listing_eligible,is_qualified
0,2020-06-29 00:00:00+00:00,9116.16,9238.00,9024.67,9192.56,42120.293261,BTC/USDT,3.871933e+08,4.582236e+08,2020-01-01 00:00:00+00:00,180,True,1
1,2020-06-30 00:00:00+00:00,9192.93,9205.00,9064.89,9138.55,31463.162801,BTC/USDT,2.875277e+08,4.557183e+08,2020-01-01 00:00:00+00:00,181,True,1
2,2020-07-01 00:00:00+00:00,9138.08,9292.00,9080.10,9232.00,38488.528699,BTC/USDT,3.553261e+08,4.522662e+08,2020-01-01 00:00:00+00:00,182,True,1
3,2020-07-02 00:00:00+00:00,9231.99,9261.96,8940.00,9086.54,45725.168076,BTC/USDT,4.154836e+08,4.484277e+08,2020-01-01 00:00:00+00:00,183,True,1
4,2020-07-03 00:00:00+00:00,9086.54,9125.00,9037.47,9058.26,28943.420177,BTC/USDT,2.621770e+08,4.345655e+08,2020-01-01 00:00:00+00:00,184,True,1


In [7]:
"""
Tạo ra Universe với tối đa 40 coin mỗi ngày,
40 coin được sắp xếp theo thanh khoản từ cao tới thấp để lọc ra những coin tốt nhất cho Universe (theo median_30)
"""
def create_universe(df, top_n = 40):
    table = df.copy()
    table["rank_by_liquid"] = table.groupby("timestamp")["median_dollar_volume_30d"].rank(method = "dense", ascending = False)
    
    table["in_rank"] = np.where((table["rank_by_liquid"] <= top_n),1,0) # có nằm trong top 40 hay k
    
    table["in_universe"] = (np.where((table["rank_by_liquid"] <= top_n) & (table["is_qualified"] == 1), 1,0))
    return table.sort_values(["timestamp", "rank_by_liquid"], ascending=[True, True]).reset_index(drop = True)

result = create_universe(result)
    

In [8]:
result

,timestamp,open,high,low,close,volume,symbol,dollar_volume,median_dollar_volume_30d,first_trading_date,listing_age_days,listing_eligible,is_qualified,rank_by_liquid,in_rank,in_universe
0,2020-06-29 00:00:00+00:00,9116.16000,9238.000000,9024.670000,9192.560000,4.212029e+04,BTC/USDT,3.871933e+08,4.582236e+08,2020-01-01 00:00:00+00:00,180,True,1,1.0,1,1
1,2020-06-29 00:00:00+00:00,224.89000,229.960000,221.260000,227.930000,5.979155e+05,ETH/USDT,1.362829e+08,1.190435e+08,2020-01-01 00:00:00+00:00,180,True,1,2.0,1,1
2,2020-06-29 00:00:00+00:00,15.37140,15.590100,15.204000,15.465600,1.511856e+06,BNB/USDT,2.338177e+07,2.825587e+07,2020-01-01 00:00:00+00:00,180,True,1,3.0,1,1
3,2020-06-29 00:00:00+00:00,0.08026,0.084440,0.080260,0.083920,3.220009e+08,ADA/USDT,2.702232e+07,2.468974e+07,2020-01-01 00:00:00+00:00,180,True,1,4.0,1,1
4,2020-06-29 00:00:00+00:00,222.67000,227.500000,218.280000,225.280000,8.267506e+04,BCH/USDT,1.862504e+07,2.035926e+07,2020-01-01 00:00:00+00:00,180,True,1,5.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
440139,2026-08-17 00:00:00+00:00,0.00046,0.000473,0.000459,0.000465,6.010048e+07,SC/USDT,2.794673e+04,6.117904e+04,2020-07-06 00:00:00+00:00,2233,True,0,312.0,0,0
440140,2026-08-17 00:00:00+00:00,0.08750,0.088500,0.087500,0.088200,1.161911e+05,GLM/USDT,1.024806e+04,5.603710e+04,2023-03-17 00:00:00+00:00,1249,True,0,313.0,0,0
440141,2026-08-17 00:00:00+00:00,0.00804,0.008330,0.008000,0.008140,1.732885e+06,QUICK/USDT,1.410568e+04,5.553868e+04,2021-08-13 00:00:00+00:00,1830,True,0,314.0,0,0
440142,2026-08-17 00:00:00+00:00,0.50600,0.510000,0.502000,0.507000,6.439664e+04,GNS/USDT,3.264910e+04,5.251430e+04,2023-02-17 00:00:00+00:00,1277,True,0,315.0,0,0


In [9]:
daily_count = result.groupby("timestamp")["symbol"].count().reset_index()
daily_count.columns = ["timestamp", "n_coins"]

print(" Số coin mỗi ngày:")
print(daily_count)

 Số coin mỗi ngày:
                     timestamp  n_coins
0    2020-06-29 00:00:00+00:00       50
1    2020-06-30 00:00:00+00:00       50
2    2020-07-01 00:00:00+00:00       51
3    2020-07-02 00:00:00+00:00       51
4    2020-07-03 00:00:00+00:00       51
...                        ...      ...
2236 2026-08-13 00:00:00+00:00      317
2237 2026-08-14 00:00:00+00:00      317
2238 2026-08-15 00:00:00+00:00      317
2239 2026-08-16 00:00:00+00:00      317
2240 2026-08-17 00:00:00+00:00      316

[2241 rows x 2 columns]


In [10]:
def summary_each_day(df):
    n_days = df["timestamp"].unique()
    table = pd.DataFrame(index = n_days)
    
    #số coin bị loại vì rank <40 
    table = df.groupby("timestamp").agg(
        n_excluded_rank = ("in_rank", lambda x: (x ==0).sum()),
        n_excluded_liquid = ("is_qualified", lambda x: (x==0).sum())
    )
    
    return table

summary_each_day(result)


,n_excluded_rank,n_excluded_liquid
timestamp,,
2020-06-29 00:00:00+00:00,10,40
2020-06-30 00:00:00+00:00,10,40
2020-07-01 00:00:00+00:00,11,41
2020-07-02 00:00:00+00:00,11,41
2020-07-03 00:00:00+00:00,11,41
...,...,...
2026-08-13 00:00:00+00:00,277,300
2026-08-14 00:00:00+00:00,277,300
2026-08-15 00:00:00+00:00,277,302


# B:Factor Construction (Feature Engineering)

Đối với những coin thuộc universe ngày t, mỗi coin đang có những characteristic/factor nào

In [11]:
result

,timestamp,open,high,low,close,volume,symbol,dollar_volume,median_dollar_volume_30d,first_trading_date,listing_age_days,listing_eligible,is_qualified,rank_by_liquid,in_rank,in_universe
0,2020-06-29 00:00:00+00:00,9116.16000,9238.000000,9024.670000,9192.560000,4.212029e+04,BTC/USDT,3.871933e+08,4.582236e+08,2020-01-01 00:00:00+00:00,180,True,1,1.0,1,1
1,2020-06-29 00:00:00+00:00,224.89000,229.960000,221.260000,227.930000,5.979155e+05,ETH/USDT,1.362829e+08,1.190435e+08,2020-01-01 00:00:00+00:00,180,True,1,2.0,1,1
2,2020-06-29 00:00:00+00:00,15.37140,15.590100,15.204000,15.465600,1.511856e+06,BNB/USDT,2.338177e+07,2.825587e+07,2020-01-01 00:00:00+00:00,180,True,1,3.0,1,1
3,2020-06-29 00:00:00+00:00,0.08026,0.084440,0.080260,0.083920,3.220009e+08,ADA/USDT,2.702232e+07,2.468974e+07,2020-01-01 00:00:00+00:00,180,True,1,4.0,1,1
4,2020-06-29 00:00:00+00:00,222.67000,227.500000,218.280000,225.280000,8.267506e+04,BCH/USDT,1.862504e+07,2.035926e+07,2020-01-01 00:00:00+00:00,180,True,1,5.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
440139,2026-08-17 00:00:00+00:00,0.00046,0.000473,0.000459,0.000465,6.010048e+07,SC/USDT,2.794673e+04,6.117904e+04,2020-07-06 00:00:00+00:00,2233,True,0,312.0,0,0
440140,2026-08-17 00:00:00+00:00,0.08750,0.088500,0.087500,0.088200,1.161911e+05,GLM/USDT,1.024806e+04,5.603710e+04,2023-03-17 00:00:00+00:00,1249,True,0,313.0,0,0
440141,2026-08-17 00:00:00+00:00,0.00804,0.008330,0.008000,0.008140,1.732885e+06,QUICK/USDT,1.410568e+04,5.553868e+04,2021-08-13 00:00:00+00:00,1830,True,0,314.0,0,0
440142,2026-08-17 00:00:00+00:00,0.50600,0.510000,0.502000,0.507000,6.439664e+04,GNS/USDT,3.264910e+04,5.251430e+04,2023-02-17 00:00:00+00:00,1277,True,0,315.0,0,0


##  B.1 — Momentum 

In [12]:
def build_momentum_factors(df, windows = [7,14,30,90]):
    table = df.copy()
    
    table["log_return"] = np.log(table["close"]) - np.log(table.groupby("symbol")["close"].shift(1)) #daily log return
    for window in windows:
        table[f"momentum_{window}d"] = table.groupby("symbol")["log_return"].transform(
            lambda x: x.rolling(window= window, min_periods = window).sum())
    
    temp = table[["timestamp", "symbol", "momentum_7d", "momentum_14d", "momentum_30d", "momentum_90d"]]
    return pd.DataFrame(temp)

result_momentum= build_momentum_factors(result)
result_momentum
    

,timestamp,symbol,momentum_7d,momentum_14d,momentum_30d,momentum_90d
0,2020-06-29 00:00:00+00:00,BTC/USDT,NaN,NaN,NaN,NaN
1,2020-06-29 00:00:00+00:00,ETH/USDT,NaN,NaN,NaN,NaN
2,2020-06-29 00:00:00+00:00,BNB/USDT,NaN,NaN,NaN,NaN
3,2020-06-29 00:00:00+00:00,ADA/USDT,NaN,NaN,NaN,NaN
4,2020-06-29 00:00:00+00:00,BCH/USDT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
440139,2026-08-17 00:00:00+00:00,SC/USDT,-0.076563,-0.076563,-0.256558,-0.674698
440140,2026-08-17 00:00:00+00:00,GLM/USDT,-0.029052,-0.054067,-0.119545,-0.388696
440141,2026-08-17 00:00:00+00:00,QUICK/USDT,-0.001228,0.028662,0.037551,-0.143920
440142,2026-08-17 00:00:00+00:00,GNS/USDT,-0.046251,0.017911,-0.180018,0.063093


In [13]:
result_momentum.isna().sum()

timestamp           0
symbol              0
momentum_7d      2702
momentum_14d     5397
momentum_30d    11540
momentum_90d    34118
dtype: int64

##  B.2 — Reversal (raw, không đảo dấu)

In [14]:
def build_reversal(df):
    table = df.copy()
    table["reversal_1d"] = np.log(table["close"]) - np.log(table.groupby("symbol")["close"].shift(1))
    table["reversal_3d"] = np.log(table["close"]) - np.log(table.groupby("symbol")["close"].shift(3))
    return table[["timestamp","symbol","reversal_1d","reversal_3d"]]

result_reversal = build_reversal(result)
result_reversal

,timestamp,symbol,reversal_1d,reversal_3d
0,2020-06-29 00:00:00+00:00,BTC/USDT,NaN,NaN
1,2020-06-29 00:00:00+00:00,ETH/USDT,NaN,NaN
2,2020-06-29 00:00:00+00:00,BNB/USDT,NaN,NaN
3,2020-06-29 00:00:00+00:00,ADA/USDT,NaN,NaN
4,2020-06-29 00:00:00+00:00,BCH/USDT,NaN,NaN
...,...,...,...,...
440139,2026-08-17 00:00:00+00:00,SC/USDT,0.008639,0.004310
440140,2026-08-17 00:00:00+00:00,GLM/USDT,0.005685,-0.003396
440141,2026-08-17 00:00:00+00:00,QUICK/USDT,0.009877,-0.001228
440142,2026-08-17 00:00:00+00:00,GNS/USDT,0.001974,-0.032981


In [15]:
result_reversal.isna().sum()

timestamp         0
symbol            0
reversal_1d     386
reversal_3d    1158
dtype: int64

##  B.4 — Volatility

In [16]:
def build_volatility(df, windows = [7,14,30]):
    table = df.copy()
    table["log_return"] = np.log(table["close"]) - np.log(table.groupby("symbol")["close"].shift(1)) 
    
    for window in windows:
        table[f"vol_{window}d"] = table.groupby("symbol")["log_return"].transform(lambda x: x.rolling
                                                                                  (window= window, min_periods = window).std())
        
    table["vol_of_vol_14d"] = table.groupby("symbol")["vol_14d"].transform(lambda x: x.rolling(window = 14,
                                                                                              min_periods = 14).std())    
    
    return table[["timestamp","symbol","vol_7d","vol_14d","vol_30d","vol_of_vol_14d"]]

build_volatility(result)

,timestamp,symbol,vol_7d,vol_14d,vol_30d,vol_of_vol_14d
0,2020-06-29 00:00:00+00:00,BTC/USDT,NaN,NaN,NaN,NaN
1,2020-06-29 00:00:00+00:00,ETH/USDT,NaN,NaN,NaN,NaN
2,2020-06-29 00:00:00+00:00,BNB/USDT,NaN,NaN,NaN,NaN
3,2020-06-29 00:00:00+00:00,ADA/USDT,NaN,NaN,NaN,NaN
4,2020-06-29 00:00:00+00:00,BCH/USDT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
440139,2026-08-17 00:00:00+00:00,SC/USDT,0.017899,0.014848,0.018866,0.004050
440140,2026-08-17 00:00:00+00:00,GLM/USDT,0.012830,0.011630,0.020922,0.007914
440141,2026-08-17 00:00:00+00:00,QUICK/USDT,0.014821,0.019258,0.018234,0.001436
440142,2026-08-17 00:00:00+00:00,GNS/USDT,0.009509,0.013317,0.021407,0.004834


##  B.5 — Build Liquid

In [17]:
def build_liquid(df, windows = [14,30]):
    table = df.copy()
    table["log_return"] = np.log(table["close"]) - np.log(table.groupby("symbol")["close"].shift(1))
    
    table["amihud_daily"] = abs(table["log_return"]) / table["dollar_volume"]
    
    for window in windows:
        table[f"amihud_{window}d"] = table.groupby("symbol")["amihud_daily"].transform(lambda x: x.rolling(window = window,
                                                                                                           min_periods = window).mean())

    return table[["timestamp","symbol","log_return","amihud_14d", "amihud_30d"]]
     
build_liquid(result)

,timestamp,symbol,log_return,amihud_14d,amihud_30d
0,2020-06-29 00:00:00+00:00,BTC/USDT,NaN,NaN,NaN
1,2020-06-29 00:00:00+00:00,ETH/USDT,NaN,NaN,NaN
2,2020-06-29 00:00:00+00:00,BNB/USDT,NaN,NaN,NaN
3,2020-06-29 00:00:00+00:00,ADA/USDT,NaN,NaN,NaN
4,2020-06-29 00:00:00+00:00,BCH/USDT,NaN,NaN,NaN
...,...,...,...,...,...
440139,2026-08-17 00:00:00+00:00,SC/USDT,0.008639,1.717671e-07,2.016798e-07
440140,2026-08-17 00:00:00+00:00,GLM/USDT,0.005685,2.666855e-07,2.322266e-07
440141,2026-08-17 00:00:00+00:00,QUICK/USDT,0.009877,3.470458e-07,2.881256e-07
440142,2026-08-17 00:00:00+00:00,GNS/USDT,0.001974,1.363408e-07,1.991994e-07


In [18]:
def build_factors(df):
    momentum = build_momentum_factors(df)
    reversal = build_reversal(df)
    vol = build_volatility(df)
    liquid = build_liquid(df)
    
    table = df.merge(momentum, on = ["timestamp","symbol"], how = 'inner').merge(reversal,
                            on = ["timestamp","symbol"], how = 'inner').merge(vol, on = ["timestamp","symbol"], how = 'inner').merge(
                                liquid, on = ['timestamp','symbol'],how = 'inner'
                            )
    return table

result_ver_1 = build_factors(result)


In [19]:
result_ver_1


,timestamp,open,high,low,close,volume,symbol,dollar_volume,median_dollar_volume_30d,first_trading_date,...,momentum_90d,reversal_1d,reversal_3d,vol_7d,vol_14d,vol_30d,vol_of_vol_14d,log_return,amihud_14d,amihud_30d
0,2020-06-29 00:00:00+00:00,9116.16000,9238.000000,9024.670000,9192.560000,4.212029e+04,BTC/USDT,3.871933e+08,4.582236e+08,2020-01-01 00:00:00+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-06-29 00:00:00+00:00,224.89000,229.960000,221.260000,227.930000,5.979155e+05,ETH/USDT,1.362829e+08,1.190435e+08,2020-01-01 00:00:00+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-06-29 00:00:00+00:00,15.37140,15.590100,15.204000,15.465600,1.511856e+06,BNB/USDT,2.338177e+07,2.825587e+07,2020-01-01 00:00:00+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-06-29 00:00:00+00:00,0.08026,0.084440,0.080260,0.083920,3.220009e+08,ADA/USDT,2.702232e+07,2.468974e+07,2020-01-01 00:00:00+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-06-29 00:00:00+00:00,222.67000,227.500000,218.280000,225.280000,8.267506e+04,BCH/USDT,1.862504e+07,2.035926e+07,2020-01-01 00:00:00+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
440139,2026-08-17 00:00:00+00:00,0.00046,0.000473,0.000459,0.000465,6.010048e+07,SC/USDT,2.794673e+04,6.117904e+04,2020-07-06 00:00:00+00:00,...,-0.674698,0.008639,0.004310,0.017899,0.014848,0.018866,0.004050,0.008639,1.717671e-07,2.016798e-07
440140,2026-08-17 00:00:00+00:00,0.08750,0.088500,0.087500,0.088200,1.161911e+05,GLM/USDT,1.024806e+04,5.603710e+04,2023-03-17 00:00:00+00:00,...,-0.388696,0.005685,-0.003396,0.012830,0.011630,0.020922,0.007914,0.005685,2.666855e-07,2.322266e-07
440141,2026-08-17 00:00:00+00:00,0.00804,0.008330,0.008000,0.008140,1.732885e+06,QUICK/USDT,1.410568e+04,5.553868e+04,2021-08-13 00:00:00+00:00,...,-0.143920,0.009877,-0.001228,0.014821,0.019258,0.018234,0.001436,0.009877,3.470458e-07,2.881256e-07
440142,2026-08-17 00:00:00+00:00,0.50600,0.510000,0.502000,0.507000,6.439664e+04,GNS/USDT,3.264910e+04,5.251430e+04,2023-02-17 00:00:00+00:00,...,0.063093,0.001974,-0.032981,0.009509,0.013317,0.021407,0.004834,0.001974,1.363408e-07,1.991994e-07


##  B.6 — Cross-sectional Standardization

In [20]:
def cal_z_score(factor_value):
    mean = factor_value.mean()
    std = factor_value.std()
    if std == 0: 
        return 0.0
    return (factor_value - mean) / std

def winsorize_series(series, lower=0.05, upper=0.95):
    lower_bound = series.quantile(lower)
    upper_bound = series.quantile(upper)
    return series.clip(lower=lower_bound, upper=upper_bound)

def build_z_score(df):
    result = df.copy()
    
    factors = [
        'momentum_7d', 'momentum_14d', 'momentum_30d', 'momentum_90d',
        'reversal_1d', 'reversal_3d',
        'vol_7d', 'vol_14d', 'vol_30d', 'vol_of_vol_14d',
        'amihud_14d', 'amihud_30d'
    ]
    mask = result['in_universe'] == 1
    
    for factor in factors:
           
        result.loc[mask, f'{factor}_winsorize'] = result.loc[mask].groupby("timestamp")[factor].transform(lambda x: winsorize_series(x))
                
        result.loc[mask, factor] = result.loc[mask].groupby("timestamp")[f'{factor}_winsorize'].transform(lambda x: cal_z_score(x))
        
        # B3: Các coin không trong universe nhận NaN
        result.loc[~mask, factor] = np.nan
        
        result.rename(columns={factor: f"z_{factor}"}, inplace=True)
        
        result.drop(columns=f'{factor}_winsorize', inplace= True)

    columns_to_drop = [
    'open','high', 'low','close','volume','dollar_volume','median_dollar_volume_30d','first_trading_date','listing_age_days',
    'listing_eligible','is_qualified','rank_by_liquid','in_rank']
    
    return result.drop(columns=columns_to_drop)

result_ver_2 = build_z_score(result_ver_1)

In [21]:
result_ver_2[~np.isnan(result_ver_2["z_momentum_90d"])]

,timestamp,symbol,in_universe,z_momentum_7d,z_momentum_14d,z_momentum_30d,z_momentum_90d,z_reversal_1d,z_reversal_3d,z_vol_7d,z_vol_14d,z_vol_30d,z_vol_of_vol_14d,log_return,z_amihud_14d,z_amihud_30d
4851,2020-09-27 00:00:00+00:00,BTC/USDT,1,0.104937,1.295379,0.608275,-0.643410,0.059951,-0.801625,-1.335920,-1.460214,-1.898988,-1.592089,0.004246,-1.130024,-1.219553
4852,2020-09-27 00:00:00+00:00,ETH/USDT,1,-0.304986,0.608550,0.439487,0.332203,0.293393,-0.199482,-0.491809,-0.556091,-0.670204,-1.220259,0.010036,-1.124777,-1.213106
4853,2020-09-27 00:00:00+00:00,LINK/USDT,1,2.119062,-0.288078,-0.790309,1.667830,1.710906,1.711488,1.784511,1.734227,1.347208,1.230560,0.045198,-0.919622,-1.009862
4854,2020-09-27 00:00:00+00:00,BNB/USDT,1,0.294696,-0.908935,1.626324,0.596841,-0.032517,1.024796,0.163254,0.335636,0.197070,-1.114051,0.001952,-0.950499,-0.984128
4855,2020-09-27 00:00:00+00:00,TRX/USDT,1,0.111595,-0.604139,1.507666,0.486506,-0.995973,-0.528470,-1.229117,-0.538236,0.136846,0.244343,-0.021946,-0.706681,-0.747670
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439835,2026-08-17 00:00:00+00:00,DOGE/USDT,1,0.212045,-0.095684,-0.067682,-1.342372,-0.233914,0.087115,0.149850,-0.209099,-0.312785,-0.235516,0.008299,-0.307855,-0.391000
439836,2026-08-17 00:00:00+00:00,TRX/USDT,1,0.162288,0.031234,0.198504,0.268907,-0.461456,-0.003640,-0.595261,-0.676579,-0.607413,-0.403616,0.003917,-0.479191,-0.546599
439838,2026-08-17 00:00:00+00:00,ADA/USDT,1,-2.028928,-1.311738,0.273445,-1.286711,-0.930979,-1.756455,-0.500855,0.008768,-0.055341,-0.008822,-0.012658,0.100633,0.075142
439839,2026-08-17 00:00:00+00:00,PUMP/USDT,1,0.864120,2.086912,1.662471,2.082405,2.306911,0.489467,1.686841,1.797505,0.699892,-0.091969,0.074697,1.698583,1.860994


In [22]:
RESEARCH_START = "2023-01-01"
RESEARCH_END = "2025-12-31"

research_z = result_ver_2[(result_ver_2["timestamp"] >= "2020-01-01") & (result_ver_2["timestamp"] <= "2021-12-31")]
research_z[(research_z["timestamp"] == "2023-01-01 00:00:00+00:00") ].head(45)

,timestamp,symbol,in_universe,z_momentum_7d,z_momentum_14d,z_momentum_30d,z_momentum_90d,z_reversal_1d,z_reversal_3d,z_vol_7d,z_vol_14d,z_vol_30d,z_vol_of_vol_14d,log_return,z_amihud_14d,z_amihud_30d


In [23]:
D_research_z = result_ver_2[(result_ver_2["timestamp"] >= '2022-01-01') & (result_ver_2["timestamp"] <= '2025-12-31')].reset_index(drop = True)
D_research_z

,timestamp,symbol,in_universe,z_momentum_7d,z_momentum_14d,z_momentum_30d,z_momentum_90d,z_reversal_1d,z_reversal_3d,z_vol_7d,z_vol_14d,z_vol_30d,z_vol_of_vol_14d,log_return,z_amihud_14d,z_amihud_30d
0,2022-01-01 00:00:00+00:00,BTC/USDT,1,-0.212788,-0.541299,0.024155,-0.267705,-0.443535,-0.484377,-1.319142,-1.427240,-1.874183,-1.142377,0.032060,-1.275887,-1.299732
1,2022-01-01 00:00:00+00:00,ETH/USDT,1,-0.550079,-0.947207,-0.029221,-0.033098,-0.702022,-0.328368,-1.295067,-1.548204,-1.747943,0.207670,0.024004,-1.275887,-1.299732
2,2022-01-01 00:00:00+00:00,BNB/USDT,1,-0.014854,-0.728972,0.053488,0.199401,-0.489813,-0.518128,-1.258354,-1.552638,-1.877973,-0.171664,0.030618,-1.272222,-1.296198
3,2022-01-01 00:00:00+00:00,LUNA/USDT,1,-0.387271,0.615426,2.051129,1.427201,0.785459,0.453285,0.080354,0.049022,1.002748,1.724150,0.070365,-1.170182,-1.143102
4,2022-01-01 00:00:00+00:00,SAND/USDT,1,-0.913554,0.226459,0.262051,2.169119,-0.770730,-0.459319,-0.765213,1.723306,0.701132,1.535183,0.021862,-1.088658,-1.123099
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317671,2025-12-31 00:00:00+00:00,GNO/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.003923,NaN,NaN
317672,2025-12-31 00:00:00+00:00,BNT/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000985,NaN,NaN
317673,2025-12-31 00:00:00+00:00,REQ/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.007752,NaN,NaN
317674,2025-12-31 00:00:00+00:00,GTC/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.015152,NaN,NaN


In [24]:
"""
Tạo ra các foward_return (sử dụng data đầu vào là result)
"""
def build_foward_return(df, windows = [1,5,14]):
    table = df.copy()
    
    table["log_return"] = np.log(table["close"]) - np.log(table.groupby("symbol")["close"].shift(1))
    for window in windows:
        table[f"fwd_{window}d"] = table.groupby("symbol")["log_return"].transform(lambda x: 
            x.rolling(window= window,min_periods = window).sum().shift(-window))
    return table[["timestamp","symbol","fwd_1d","fwd_5d","fwd_14d"]]
foward_return = build_foward_return(result)

foward_return
    

,timestamp,symbol,fwd_1d,fwd_5d,fwd_14d
0,2020-06-29 00:00:00+00:00,BTC/USDT,-0.005893,-0.006231,0.005431
1,2020-06-29 00:00:00+00:00,ETH/USDT,-0.010275,0.005426,0.049515
2,2020-06-29 00:00:00+00:00,BNB/USDT,-0.005303,0.005751,0.173982
3,2020-06-29 00:00:00+00:00,ADA/USDT,-0.011144,0.175806,0.392593
4,2020-06-29 00:00:00+00:00,BCH/USDT,-0.014126,-0.000089,0.030212
...,...,...,...,...,...
440139,2026-08-17 00:00:00+00:00,SC/USDT,NaN,NaN,NaN
440140,2026-08-17 00:00:00+00:00,GLM/USDT,NaN,NaN,NaN
440141,2026-08-17 00:00:00+00:00,QUICK/USDT,NaN,NaN,NaN
440142,2026-08-17 00:00:00+00:00,GNS/USDT,NaN,NaN,NaN


In [25]:
# factor_Construction là bảng z_score ứng với thời gian từ 2020-2021
factor_construction = research_z.merge(foward_return, on = ["timestamp","symbol"], how = 'left')
factor_construction
factor_construction.to_parquet('B1_factor_construction.parquet')



In [26]:
# D_factor_construction là bảng từ 2022-2025 để phục vụ cho phần D của notebook thứ 2
D_factor_construction = D_research_z.merge(foward_return, on = ["timestamp","symbol"], how = 'left')
D_factor_construction 
#D_factor_construction.to_parquet('2022-2025_factor_construction.parquet')


,timestamp,symbol,in_universe,z_momentum_7d,z_momentum_14d,z_momentum_30d,z_momentum_90d,z_reversal_1d,z_reversal_3d,z_vol_7d,z_vol_14d,z_vol_30d,z_vol_of_vol_14d,log_return,z_amihud_14d,z_amihud_30d,fwd_1d,fwd_5d,fwd_14d
0,2022-01-01 00:00:00+00:00,BTC/USDT,1,-0.212788,-0.541299,0.024155,-0.267705,-0.443535,-0.484377,-1.319142,-1.427240,-1.874183,-1.142377,0.032060,-1.275887,-1.299732,-0.009188,-0.102294,-0.102248
1,2022-01-01 00:00:00+00:00,ETH/USDT,1,-0.550079,-0.947207,-0.029221,-0.033098,-0.702022,-0.328368,-1.295067,-1.548204,-1.747943,0.207670,0.024004,-1.275887,-1.299732,0.016522,-0.100115,-0.124109
2,2022-01-01 00:00:00+00:00,BNB/USDT,1,-0.014854,-0.728972,0.053488,0.199401,-0.489813,-0.518128,-1.258354,-1.552638,-1.877973,-0.171664,0.030618,-1.272222,-1.296198,0.006992,-0.109520,-0.064222
3,2022-01-01 00:00:00+00:00,LUNA/USDT,1,-0.387271,0.615426,2.051129,1.427201,0.785459,0.453285,0.080354,0.049022,1.002748,1.724150,0.070365,-1.170182,-1.143102,-0.024734,-0.156462,-0.050324
4,2022-01-01 00:00:00+00:00,SAND/USDT,1,-0.913554,0.226459,0.262051,2.169119,-0.770730,-0.459319,-0.765213,1.723306,0.701132,1.535183,0.021862,-1.088658,-1.123099,-0.009780,-0.131208,-0.209213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317671,2025-12-31 00:00:00+00:00,GNO/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.003923,NaN,NaN,0.024672,0.105572,0.146774
317672,2025-12-31 00:00:00+00:00,BNT/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000985,NaN,NaN,0.012474,0.084218,0.092330
317673,2025-12-31 00:00:00+00:00,REQ/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.007752,NaN,NaN,0.017225,0.055387,0.042519
317674,2025-12-31 00:00:00+00:00,GTC/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.015152,NaN,NaN,0.073563,0.161755,0.094616


Tổng kết:

- B1_factor_construction.parquet: là dataset phục vụ cho các kiểm định thống kê để tìm ra những factor có ý nghĩa nhất (được trình bày ở notebook thứ 2), thời gian cho kiểm định là từ 2020-2021 để không look-ahead trước phần tương lai của 2023-2025 mà ta sẽ thực hiện chiến lược
  
- 2022-2025_factor_construction.parquet là dataset phục vụ cho việc xây dựng các trọng số (weight) ở notebook thứ 3, thời gian để xây dựng các trọng số là từ 2023-2025
